# XGBoost & LightGBM: Where Machine Learning Beats Traditional Actuarial Models

**Author:** Andres Gonzalez Ortega | **Date:** March 2026

---

## Why Gradient Boosting for Insurance Pricing?

Tree-based ensemble models have several advantages over GLMs for pricing:

1. **Automatic interaction detection**: Trees naturally capture VehAge x DrivAge interactions without manual specification
2. **Non-linear effects**: No need to manually bin continuous variables -- the model finds optimal splits
3. **Robustness**: Less sensitive to outliers and distributional assumptions
4. **Feature importance**: Built-in ranking of what matters most

### The Experiment Pipeline

We use a modern ML engineering stack:
- **Optuna** for Bayesian hyperparameter optimization (smarter than grid search)
- **MLflow** for experiment tracking (reproducibility and comparison)
- **SHAP** for model interpretability (TreeSHAP is exact and fast)

> *En el contexto del mercado mexicano, los modelos de ML aun no son ampliamente usados en tarificacion. Las aseguradoras grandes (GNP, Qualitas, AXA Mexico) estan empezando a explorar GBMs como complemento a los GLMs tradicionales. Este notebook muestra como hacerlo correctamente.*

### Key Design Decisions

- **Poisson objective**: Both XGBoost and LightGBM support `count:poisson` / `poisson` objectives natively
- **Exposure as sample weight**: Tree models do not have a natural offset mechanism, so we use exposure as sample weight
- **Frequency-severity separation**: Same decomposition as GLMs -- separate models for how often and how much

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0] if Path().resolve().name == 'notebooks' else Path().resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT.parent.parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns
import matplotlib.pyplot as plt

pio.templates.default = "plotly_white"
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

import xgboost as xgb
import lightgbm as lgb
import shap

## 1. Data Preparation

In [ ]:
from scripts.utils.data_loader import load_merged, split_data
from feature_engineering import create_features, get_gbm_feature_names

# Load and split
df = load_merged()
train, val, test = split_data(df)

# Create GBM features (label-encoded categoricals)
train_gbm = create_features(train, for_glm=False)
val_gbm = create_features(val, for_glm=False)
test_gbm = create_features(test, for_glm=False)

feature_names = get_gbm_feature_names(train_gbm)
print(f"Number of GBM features: {len(feature_names)}")
print(f"Features: {feature_names}")

## 2. XGBoost Frequency Model with Optuna

In [ ]:
from gbm_models import train_xgb_frequency, predict_xgb

# Train with Optuna hyperparameter search
# Reduce n_trials for notebook demo (use 100+ for production)
xgb_model, xgb_params = train_xgb_frequency(
    X_train=train_gbm,
    y_train=train_gbm['ClaimNb'],
    exposure_train=train_gbm['Exposure'],
    X_val=val_gbm,
    y_val=val_gbm['ClaimNb'],
    exposure_val=val_gbm['Exposure'],
    feature_names=feature_names,
    n_trials=50,  # Increase to 100+ for production
)

## 3. Best Parameters

In [ ]:
print("XGBoost -- Best Hyperparameters:")
for k, v in sorted(xgb_params.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")

### So What?
Optuna uses **Tree-structured Parzen Estimators (TPE)** to efficiently search the hyperparameter space.
Key parameters to watch:
- **max_depth**: Controls tree complexity. Deep trees capture more interactions but risk overfitting.
- **learning_rate**: Lower = more robust but slower. Optuna finds the sweet spot.
- **min_child_weight**: Prevents overfitting on rare segments (important for insurance with sparse claims).
- **reg_alpha/reg_lambda**: L1/L2 regularization. Critical for insurance where we want conservative estimates.

## 4. LightGBM Frequency Model

In [ ]:
from gbm_models import train_lgb_frequency, predict_lgb

lgb_model, lgb_params = train_lgb_frequency(
    X_train=train_gbm,
    y_train=train_gbm['ClaimNb'],
    exposure_train=train_gbm['Exposure'],
    X_val=val_gbm,
    y_val=val_gbm['ClaimNb'],
    exposure_val=val_gbm['Exposure'],
    feature_names=feature_names,
    n_trials=50,
)

print("\nLightGBM -- Best Hyperparameters:")
for k, v in sorted(lgb_params.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")

## 5. XGBoost Severity Model

In [ ]:
from gbm_models import train_xgb_severity

xgb_sev_model, xgb_sev_params = train_xgb_severity(
    X_train=train_gbm,
    y_train=train_gbm['AvgClaimAmount'],
    X_val=val_gbm,
    y_val=val_gbm['AvgClaimAmount'],
    feature_names=feature_names,
    n_trials=30,
)

## 6. Feature Importance Comparison

In [ ]:
from gbm_models import get_feature_importance

xgb_imp = pd.DataFrame(get_feature_importance(xgb_model, feature_names, 'xgb'))
lgb_imp = pd.DataFrame(get_feature_importance(lgb_model, feature_names, 'lgb'))

# Side by side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# XGBoost
xgb_top = xgb_imp.head(12)
axes[0].barh(xgb_top['feature'], xgb_top['importance'], color='#2E86AB')
axes[0].set_title('XGBoost Feature Importance (Gain)')
axes[0].invert_yaxis()

# LightGBM
lgb_top = lgb_imp.head(12)
axes[1].barh(lgb_top['feature'], lgb_top['importance'], color='#A23B72')
axes[1].set_title('LightGBM Feature Importance (Gain)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### So What?
Both models agree on the top features but may disagree on the ranking:
- **BonusMalus** is typically the most important (consistent with GLM findings)
- **DrivAge** and **Density/LogDensity** are consistently in the top 5
- **VehAge** and **VehPower** are important but secondary

The key difference from the GLM: tree models use the *raw continuous values* rather than bins,
allowing them to find the optimal splits themselves. This is where they often find patterns the
actuary's manual binning misses.

## 7. Test Set Predictions

In [ ]:
from scripts.utils.evaluation import compute_all_metrics

# XGBoost predictions
y_test = test_gbm['ClaimNb']
exposure_test = test_gbm['Exposure']

xgb_pred = predict_xgb(xgb_model, test_gbm, feature_names)
lgb_pred = predict_lgb(lgb_model, test_gbm, feature_names)

print("Test Set Metrics:")
print("\n--- XGBoost ---")
xgb_metrics = compute_all_metrics(y_test, xgb_pred, exposure_test, family='poisson')
for k, v in xgb_metrics.items():
    print(f"  {k}: {v}")

print("\n--- LightGBM ---")
lgb_metrics = compute_all_metrics(y_test, lgb_pred, exposure_test, family='poisson')
for k, v in lgb_metrics.items():
    print(f"  {k}: {v}")

## 8. Metrics Comparison Table (All Models)

In [ ]:
from scripts.utils.evaluation import model_comparison_table

# Load GLM predictions for comparison (retrain quickly or load from saved)
from feature_engineering import create_features as create_feat_fn, get_glm_feature_names
from glm_models import train_poisson_glm, predict_glm

test_glm_df = create_feat_fn(test, for_glm=True)
train_glm_df = create_feat_fn(train, for_glm=True)
glm_features = get_glm_feature_names(train_glm_df)

poisson_glm = train_poisson_glm(
    train_glm_df, train_glm_df['ClaimNb'], train_glm_df['Exposure'], glm_features
)
glm_pred = predict_glm(poisson_glm, test_glm_df, glm_features, exposure=test_glm_df['Exposure'])

# Compare all three
comparison = model_comparison_table(
    y_test,
    {'Poisson GLM': glm_pred, 'XGBoost': xgb_pred, 'LightGBM': lgb_pred},
    sample_weight=exposure_test,
    family='poisson',
)
comparison_df = pd.DataFrame(comparison)
display(comparison_df)

### So What?
The comparison table reveals the **marginal improvement** from ML over the traditional GLM.
Typical findings on freMTPL2:
- GBMs improve Gini by 2-5 points over the GLM
- The improvement in Poisson deviance is smaller (1-3%)
- LightGBM and XGBoost perform similarly (within noise)

The improvement may seem modest, but in insurance pricing, **every point of Gini matters**: better risk
separation means fewer cross-subsidies, fairer pricing, and better competitive positioning.

## 9. SHAP Analysis -- Model Interpretability

In [ ]:
from shap_analysis import compute_all_shap_outputs

shap_outputs = compute_all_shap_outputs(
    xgb_model, test_gbm, feature_names,
    model_type='xgb',
    dependence_features=['BonusMalus', 'DrivAge', 'VehAge'],
    sample_size=10000,
)

print(f"SHAP summary computed for {len(shap_outputs['summary'])} features")
print(f"Dependence plots for: {list(shap_outputs['dependence'].keys())}")

## 10. SHAP Summary Bar Chart

In [ ]:
summary_df = pd.DataFrame(shap_outputs['summary'])

fig = px.bar(summary_df, y='feature', x='mean_abs_shap',
             orientation='h',
             title='SHAP Feature Importance (Top 15) -- What Drives XGBoost Predictions?',
             labels={'mean_abs_shap': 'Mean |SHAP Value|', 'feature': 'Feature'},
             color_discrete_sequence=['#2E86AB'])
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
fig.show()

## 11. SHAP Dependence Plots -- Non-Linear Effects

In [ ]:
dep_features = ['BonusMalus', 'DrivAge', 'VehAge']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feat in enumerate(dep_features):
    dep_data = shap_outputs['dependence'][feat]
    points = pd.DataFrame(dep_data['points'])

    axes[i].scatter(points['x'], points['shap_value'], alpha=0.3, s=3, c='#2E86AB')
    axes[i].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('SHAP Value')
    axes[i].set_title(f'SHAP Dependence: {feat}')

plt.suptitle('Non-Linear Effects Captured by XGBoost', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### So What? -- The Key Insight

The SHAP dependence plots reveal the **non-linear patterns** that tree models capture:

1. **BonusMalus**: Near-linear positive effect, but with a steeper slope above 100. The model aggressively surcharges drivers with the worst records.

2. **DrivAge**: The **U-shaped curve** is clearly visible in SHAP values. Young drivers (18-25) have positive SHAP values (higher risk), middle-aged have negative (lower risk), and elderly see an uptick. A linear GLM coefficient cannot capture this -- it requires manual binning.

3. **VehAge**: A more complex pattern. Very new cars (0-2 years) and very old cars (15+) may show different effects. The model finds the optimal thresholds without manual intervention.

This is the fundamental value proposition of GBMs for pricing: **they find patterns the actuary did not think to look for.**

> *Para las aseguradoras mexicanas que estan evaluando ML para tarificacion, los graficos SHAP son la herramienta clave para "explicar" el modelo a los reguladores y a los suscriptores. La CNSF podria aceptar un modelo GBM si se demuestra que cada efecto tiene justificacion actuarial, y SHAP proporciona exactamente eso.*

## 12. MLflow Experiment Summary

In [ ]:
import mlflow

# List experiments
print("MLflow Experiments:")
for exp in mlflow.search_experiments():
    print(f"  - {exp.name} (ID: {exp.experiment_id})")

# Show best runs for XGBoost frequency
try:
    runs = mlflow.search_runs(
        experiment_names=["insurance_pricing_xgb_freq"],
        order_by=["metrics.val_poisson_deviance ASC"],
        max_results=5,
    )
    if len(runs) > 0:
        print("\nTop 5 XGBoost Frequency Runs:")
        display(runs[['params.max_depth', 'params.learning_rate',
                       'metrics.val_poisson_deviance', 'metrics.best_iteration']].head())
except Exception as e:
    print(f"MLflow query failed (expected if no runs yet): {e}")

---

## Summary

| Aspect | XGBoost | LightGBM |
|--------|---------|----------|
| Training speed | Moderate | Fast |
| Hyperparameters | Optuna (50 trials) | Optuna (50 trials) |
| Feature handling | Label-encoded | Label-encoded |
| Key advantage | Robust, well-studied | Faster, handles categoricals natively |
| SHAP support | TreeSHAP (exact) | TreeSHAP (exact) |

**Key Finding:** GBMs improve risk discrimination over the GLM baseline by capturing non-linear effects and interactions that manual feature engineering misses.

**Next notebook: 04_model_comparison.ipynb** -- Head-to-head evaluation and business implications.